In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 3-Class Disaster Triage: Balanced LMNN Metric Learning + Random Forest (`models/train_distant_analysis.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and executes **Large Margin Nearest Neighbor (LMNN)** distance metric learning from `metric_learn` across the 8 core arrival triage features using **balanced metric sampling**, followed by a **Random Forest Classifier** (with no class weighting) to classify patients across all 3 disaster triage tiers:

```mermaid
flowchart TD
    Raw["Raw Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> RUS["RandomUnderSampler (Balanced 3-Class Split: ~3.7k RED, ~3.7k YELLOW, ~3.7k GREEN)"]
    RUS --> LMNN["3-Class Large Margin Nearest Neighbor (LMNN from metric_learn: R^8 -> R^2)"]
    LMNN --> Manifold["2D LMNN Metric Manifold Coordinates (z1: LMNN Dim 1, z2: LMNN Dim 2)"]
    Manifold --> RF["Random Forest Classifier (Ensemble on 2D LMNN Space, class_weight=None)"]
    RF --> Eval["Holdout Test Evaluation (83k Visits) & 3-Class Decision Boundary Contours"]
```

### 🎯 3-Tier Disaster Triage Acuity Mapping
1. **`Tier 0: RED (ESI 1)`** ($y=0$): Immediate Resuscitation / Life Threat ($5,271$ visits, $\sim 0.94\%$).
2. **`Tier 1: YELLOW (ESI 2–3)`** ($y=1$): Emergent & Urgent conditions ($440,059$ visits, $\sim 78.86\%$).
3. **`Tier 2: GREEN (ESI 4–5)`** ($y=2$): Semi-urgent & Non-urgent conditions ($112,699$ visits, $\sim 20.19\%$).

### 🩺 8 Core Arrival Triage Features
1. `age`
2. `cc_breathingdifficulty`
3. `gender` (0 = Female, 1 = Male)
4. `triage_vital_hr` (Heart Rate)
5. `triage_vital_sbp` (Systolic Blood Pressure)
6. `triage_vital_dbp` (Diastolic Blood Pressure)
7. `triage_vital_rr` (Respiratory Rate)
8. `triage_vital_o2` (Oxygen Saturation - SpO2)

### 📐 Why Balanced LMNN + Random Forest?
1. **Multi-Class Metric Learning**: LMNN learns a 2D linear transformation matrix $L \in \mathbb{R}^{2 \times 8}$ that simultaneously pulls same-acuity clinical neighbors together and enforces large margins between RED, YELLOW, and GREEN cohorts.
2. **Balanced Metric Optimization**: Fitting LMNN on equal numbers of RED, YELLOW, and GREEN training cases prevents majority YELLOW visits from overwhelming target neighbor selection.
3. **Ensemble Non-Linear Partitions**: Random Forest on the 2D LMNN metric space non-linearly isolates the complex 3-tier boundary without artificial class weighting.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data from R, Partition & Apply Preprocessing
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from imblearn.under_sampling import RandomUnderSampler
from metric_learn import LMNN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# 3-Tier Categorical Acuity Labels:
# 0: RED (ESI 1), 1: YELLOW (ESI 2-3), 2: GREEN (ESI 4-5)
y_3tier_all = np.zeros(len(esi_all), dtype=np.int32)
y_3tier_all[esi_all == 1] = 0                          # Tier 0: RED (ESI 1)
y_3tier_all[np.isin(esi_all, [2, 3])] = 1              # Tier 1: YELLOW (ESI 2-3)
y_3tier_all[np.isin(esi_all, [4, 5])] = 2              # Tier 2: GREEN (ESI 4-5)
TIER_LABELS = ['RED (ESI 1)', 'YELLOW (ESI 2-3)', 'GREEN (ESI 4-5)']

print("=========================================================================")
print("     5v_cleandf 3-TIER DISASTER TRIAGE COHORT (LMNN + RANDOM FOREST)")
print("=========================================================================")
print(f"Total Valid ESI Visits: {len(esi_all):,}")
print(f"  * Tier 0 [RED (ESI 1)]     : {np.sum(y_3tier_all == 0):,} ({np.mean(y_3tier_all == 0)*100:.2f}%)")
print(f"  * Tier 1 [YELLOW (ESI 2-3)]: {np.sum(y_3tier_all == 1):,} ({np.mean(y_3tier_all == 1)*100:.2f}%)")
print(f"  * Tier 2 [GREEN (ESI 4-5)] : {np.sum(y_3tier_all == 2):,} ({np.mean(y_3tier_all == 2)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(esi_all)), test_size=0.30, stratify=y_3tier_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_3tier_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_3tier_all[itr]
y_val   = y_3tier_all[iva]
y_test  = y_3tier_all[ite]

# Fit SimpleImputer & StandardScaler strictly on Training partition
print("Fitting SimpleImputer(strategy='median') & StandardScaler on Training set...")
imputer = SimpleImputer(strategy='median')
X_tr_imp  = imputer.fit_transform(raw_tr)
X_val_imp = imputer.transform(raw_val)
X_te_imp  = imputer.transform(raw_te)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_tr_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_te_imp)

print(f"✓ Partition Shapes: Train={X_train_scaled.shape}, Val={X_val_scaled.shape}, Test={X_test_scaled.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Balanced 3-Class LMNN Metric Learning Optimization
# ---------------------------------------------------------------------------
print("Applying Balanced 3-Class Downsampling for LMNN Metric Learning...")

# Balance all 3 classes in the training split so that LMNN optimizes equal margins
rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_tr_res, y_tr_res = rus.fit_resample(X_train_scaled, y_train)

print(f"✓ Balanced LMNN Training Cohort: {len(y_tr_res):,} visits")
print(f"  * RED (ESI 1)    : {np.sum(y_tr_res == 0):,} visits")
print(f"  * YELLOW (ESI 2-3): {np.sum(y_tr_res == 1):,} visits")
print(f"  * GREEN (ESI 4-5) : {np.sum(y_tr_res == 2):,} visits")

print("\nFitting 3-Class Large Margin Nearest Neighbor (LMNN: R^8 -> R^2)...")
lmnn = LMNN(
    k=5,
    n_components=2,
    max_iter=100,
    random_state=42,
    verbose=True
)
lmnn.fit(X_tr_res, y_tr_res)

L_mat = lmnn.components_
print(f"\n✓ Learned LMNN Metric Transformation Matrix L Shape: {L_mat.shape}")

# Transform all dataset partitions through learned LMNN metric space
print("Projecting 8-D Arrival Vitals into 2-D LMNN Metric Space...")
Z_train = lmnn.transform(X_train_scaled)
Z_val   = lmnn.transform(X_val_scaled)
Z_test  = lmnn.transform(X_test_scaled)
Z_tr_res = lmnn.transform(X_tr_res)

# 2D Linear PCA of Raw 8 Features (for Before-Projection Baseline)
pca_raw = PCA(n_components=2, random_state=42)
X_raw_pca_te = pca_raw.fit_transform(X_test_scaled)

print(f"✓ 2D Metric Space Shapes: Train={Z_train.shape}, Val={Z_val.shape}, Test={Z_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Train Random Forest on the 2D LMNN Metric Space (No Class Weighting)
# ---------------------------------------------------------------------------
print("Training Random Forest Classifier on 2D LMNN Metric Space (class_weight=None)...")

# Train Random Forest without class weighting on the balanced LMNN metric representations
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight=None,  # Class weighting removed per user request
    random_state=42,
    n_jobs=-1
)
rf_model.fit(Z_tr_res, y_tr_res)

# Validation split check
pred_val = rf_model.predict(Z_val)
val_acc  = accuracy_score(y_val, pred_val)
val_bacc = balanced_accuracy_score(y_val, pred_val)

print(f"✓ Random Forest Successfully Trained!")
print(f"  * Estimators        : {rf_model.n_estimators}")
print(f"  * Max Depth         : {rf_model.max_depth}")
print(f"  * Class Weighting   : None (Natural Frequencies)")
print(f"  * Validation Accuracy: {val_acc*100:.2f}% | Balanced Accuracy: {val_bacc*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Holdout Test Set Evaluation of 3-Class LMNN + Random Forest (83,705 Visits)
# ---------------------------------------------------------------------------
pred_test = rf_model.predict(Z_test)
p_test    = rf_model.predict_proba(Z_test)

# 3-Class Metrics Computation
acc_3tier     = accuracy_score(y_test, pred_test)
bal_acc_3tier = balanced_accuracy_score(y_test, pred_test)
rec_per_class = recall_score(y_test, pred_test, average=None)
prec_per_class= precision_score(y_test, pred_test, average=None, zero_division=0)
f1_per_class  = f1_score(y_test, pred_test, average=None, zero_division=0)
macro_f1      = f1_score(y_test, pred_test, average='macro', zero_division=0)
weighted_f1   = f1_score(y_test, pred_test, average='weighted', zero_division=0)

report_rows = [
    {
        'Triage_Tier': 'Tier 0: RED (ESI 1)',
        'True_Visits': int(np.sum(y_test == 0)),
        'Predicted_Visits': int(np.sum(pred_test == 0)),
        'Sensitivity (Recall)': round(rec_per_class[0], 4),
        'Precision': round(prec_per_class[0], 4),
        'F1_Score': round(f1_per_class[0], 4)
    },
    {
        'Triage_Tier': 'Tier 1: YELLOW (ESI 2-3)',
        'True_Visits': int(np.sum(y_test == 1)),
        'Predicted_Visits': int(np.sum(pred_test == 1)),
        'Sensitivity (Recall)': round(rec_per_class[1], 4),
        'Precision': round(prec_per_class[1], 4),
        'F1_Score': round(f1_per_class[1], 4)
    },
    {
        'Triage_Tier': 'Tier 2: GREEN (ESI 4-5)',
        'True_Visits': int(np.sum(y_test == 2)),
        'Predicted_Visits': int(np.sum(pred_test == 2)),
        'Sensitivity (Recall)': round(rec_per_class[2], 4),
        'Precision': round(prec_per_class[2], 4),
        'F1_Score': round(f1_per_class[2], 4)
    }
]

report_df = pd.DataFrame(report_rows)
print("=====================================================================================================================")
print("    HOLDOUT TEST EVALUATION: BALANCED LMNN METRIC LEARNING + RANDOM FOREST (3-TIER)")
print("=====================================================================================================================")
print(f"Total Test Cohort Evaluated: {len(y_test):,} visits")
print(f"Overall Accuracy         : {acc_3tier*100:.2f}%")
print(f"Macro Balanced Accuracy  : {bal_acc_3tier*100:.2f}%")
print(f"Macro F1-Score           : {macro_f1:.4f}")
print(f"Weighted F1-Score        : {weighted_f1:.4f}\n")
print(report_df.to_string(index=False))
print("=====================================================================================================================\n")

print("Detailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=TIER_LABELS, digits=4))

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'lmnn_random_forest_3class_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Metrics report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 3-Class Confusion Matrix Heatmap for LMNN + Random Forest
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8.5, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm,
    annot=annot,
    fmt='',
    cmap='Blues',
    cbar=True,
    ax=ax,
    vmin=0,
    vmax=1,
    xticklabels=TIER_LABELS,
    yticklabels=TIER_LABELS
)

ax.set_title(
    f'Balanced LMNN + Random Forest: 3-Class Confusion Matrix (Holdout Test)\n'
    f'Accuracy: {acc_3tier*100:.2f}% | Macro Balanced Accuracy: {bal_acc_3tier*100:.2f}%',
    fontsize=11.5,
    fontweight='bold',
    pad=12
)
ax.set_xlabel('Predicted Disaster Triage Tier', fontsize=11, fontweight='bold')
ax.set_ylabel('True Disaster Triage Tier', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'distant_analysis', 'lmnn_rf_3class_confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'lmnn_rf_3class_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: BEFORE vs AFTER Scatter Plots & Random Forest 3-Class Boundaries
# ---------------------------------------------------------------------------
np.random.seed(42)
test_red_idx    = np.where(y_test == 0)[0]
test_yellow_idx = np.where(y_test == 1)[0]
test_green_idx  = np.where(y_test == 2)[0]

test_yellow_sample = np.random.choice(test_yellow_idx, min(1500, len(test_yellow_idx)), replace=False)
test_green_sample  = np.random.choice(test_green_idx, min(1500, len(test_green_idx)), replace=False)

fig, axes = plt.subplots(2, 2, figsize=(17, 14))

# Panel 1: BEFORE - Heart Rate vs Systolic Blood Pressure
hr_idx  = FEATURES.index('triage_vital_hr')
sbp_idx = FEATURES.index('triage_vital_sbp')

axes[0, 0].scatter(X_te_imp[test_green_sample, hr_idx], X_te_imp[test_green_sample, sbp_idx],
                   c='#2ca02c', alpha=0.45, s=16, label='GREEN (ESI 4-5)')
axes[0, 0].scatter(X_te_imp[test_yellow_sample, hr_idx], X_te_imp[test_yellow_sample, sbp_idx],
                   c='#ff7f0e', alpha=0.40, s=16, label='YELLOW (ESI 2-3)')
axes[0, 0].scatter(X_te_imp[test_red_idx, hr_idx], X_te_imp[test_red_idx, sbp_idx],
                   c='#d62728', alpha=0.85, s=34, edgecolors='black', linewidth=0.5, label='RED (ESI 1)')
axes[0, 0].set_title('BEFORE: Raw Vitals (Heart Rate vs Systolic BP)\n[Heavy 3-Tier Overlap Across Acuity Levels]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Heart Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_ylabel('Systolic BP (mmHg)', fontsize=10.5, fontweight='bold')
axes[0, 0].set_xlim(30, 200)
axes[0, 0].set_ylim(50, 240)
axes[0, 0].grid(True, linestyle=':', alpha=0.4)
axes[0, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 2: BEFORE - Respiratory Rate vs SpO2
rr_idx = FEATURES.index('triage_vital_rr')
o2_idx = FEATURES.index('triage_vital_o2')

axes[0, 1].scatter(X_te_imp[test_green_sample, rr_idx], X_te_imp[test_green_sample, o2_idx],
                   c='#2ca02c', alpha=0.45, s=16, label='GREEN (ESI 4-5)')
axes[0, 1].scatter(X_te_imp[test_yellow_sample, rr_idx], X_te_imp[test_yellow_sample, o2_idx],
                   c='#ff7f0e', alpha=0.40, s=16, label='YELLOW (ESI 2-3)')
axes[0, 1].scatter(X_te_imp[test_red_idx, rr_idx], X_te_imp[test_red_idx, o2_idx],
                   c='#d62728', alpha=0.85, s=34, edgecolors='black', linewidth=0.5, label='RED (ESI 1)')
axes[0, 1].set_title('BEFORE: Raw Vitals (Respiratory Rate vs SpO2)\n[Entangled Respiratory and Oxygenation Profiles]', fontsize=11.5, fontweight='bold', pad=10)
axes[0, 1].set_xlabel('Respiratory Rate (bpm)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_ylabel('Oxygen Saturation SpO2 (%)', fontsize=10.5, fontweight='bold')
axes[0, 1].set_xlim(6, 45)
axes[0, 1].set_ylim(70, 100)
axes[0, 1].grid(True, linestyle=':', alpha=0.4)
axes[0, 1].legend(loc='lower left', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 3: BEFORE - 2D Linear PCA of Raw 8 Features
axes[1, 0].scatter(X_raw_pca_te[test_green_sample, 0], X_raw_pca_te[test_green_sample, 1],
                   c='#2ca02c', alpha=0.45, s=16, label='GREEN (ESI 4-5)')
axes[1, 0].scatter(X_raw_pca_te[test_yellow_sample, 0], X_raw_pca_te[test_yellow_sample, 1],
                   c='#ff7f0e', alpha=0.40, s=16, label='YELLOW (ESI 2-3)')
axes[1, 0].scatter(X_raw_pca_te[test_red_idx, 0], X_raw_pca_te[test_red_idx, 1],
                   c='#d62728', alpha=0.85, s=34, edgecolors='black', linewidth=0.5, label='RED (ESI 1)')
axes[1, 0].set_title('BEFORE: 2D Linear PCA Baseline (Original 8-D Space)\n[Unsupervised Linear Projection Fails to Separate 3 Tiers]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 0].set_xlabel('Linear Principal Component 1', fontsize=10.5, fontweight='bold')
axes[1, 0].set_ylabel('Linear Principal Component 2', fontsize=10.5, fontweight='bold')
axes[1, 0].grid(True, linestyle=':', alpha=0.4)
axes[1, 0].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

# Panel 4: AFTER - 2D Balanced LMNN Metric Space + Random Forest 3-Class Boundaries
z1_min, z1_max = Z_test[:, 0].min() - 0.5, Z_test[:, 0].max() + 0.5
z2_min, z2_max = Z_test[:, 1].min() - 0.5, Z_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(z1_min, z1_max, 250), np.linspace(z2_min, z2_max, 250))
grid_z = np.c_[xx.ravel(), yy.ravel()]

# Random Forest 3-Class Predicted Labels on Grid
preds_grid = rf_model.predict(grid_z).reshape(xx.shape)
cmap_3class = plt.cm.get_cmap('Set1', 3)
axes[1, 1].contourf(xx, yy, preds_grid, levels=[-0.5, 0.5, 1.5, 2.5], colors=['#ffcccc', '#ffe6cc', '#ccffcc'], alpha=0.65)
axes[1, 1].contour(xx, yy, preds_grid, levels=[0.5, 1.5], colors='black', linewidths=1.8, linestyles='--')

# Overlay Test Scatter Points
axes[1, 1].scatter(Z_test[test_green_sample, 0], Z_test[test_green_sample, 1],
                   c='#2ca02c', alpha=0.45, s=16, label='GREEN (ESI 4-5)')
axes[1, 1].scatter(Z_test[test_yellow_sample, 0], Z_test[test_yellow_sample, 1],
                   c='#ff7f0e', alpha=0.40, s=16, label='YELLOW (ESI 2-3)')
axes[1, 1].scatter(Z_test[test_red_idx, 0], Z_test[test_red_idx, 1],
                   c='#d62728', alpha=0.85, s=36, edgecolors='black', linewidth=0.5, label='RED (ESI 1)')

axes[1, 1].set_title('AFTER: 2D Balanced LMNN Space + Random Forest 3-Class Boundaries\n[Metric Learn LMNN (k=5) + Random Forest Classifier (class_weight=None)]', fontsize=11.5, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Component 1: LMNN Metric Dimension 1 (z1)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_ylabel('Component 2: LMNN Metric Dimension 2 (z2)', fontsize=10.5, fontweight='bold')
axes[1, 1].set_xlim(z1_min, z1_max)
axes[1, 1].set_ylim(z2_min, z2_max)
axes[1, 1].grid(True, linestyle=':', alpha=0.4)
axes[1, 1].legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

plt.suptitle('3-Class Disaster Triage: Balanced LMNN Metric Learning & Random Forest Classification\nBEFORE vs AFTER Comparison on Holdout Test Set (RED vs YELLOW vs GREEN)', fontsize=14.5, fontweight='bold', y=0.995)
plt.tight_layout()

scatter_file = os.path.join(plots_dir, 'distant_analysis', 'lmnn_rf_3class_before_after_scatter.png')
plt.savefig(scatter_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'lmnn_rf_3class_before_after_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Before/After 3-Class LMNN RF scatter plot saved to: {scatter_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Export Production 3-Class LMNN + Random Forest Bundle & Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

lmnn_rf_bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'lmnn': lmnn,
    'L_matrix': L_mat,
    'rf_model': rf_model,
    'features': FEATURES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'lmnn_random_forest_3class_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(lmnn_rf_bundle, f)

manifest = dict(
    pipeline_architecture='3Class_Balanced_LMNN_Metric_Learning_Plus_Random_Forest',
    metric_method='Large_Margin_Nearest_Neighbor_LMNN_3Class',
    lmnn_k_neighbors=5,
    lmnn_output_dimensions=2,
    transformation_matrix_shape=list(L_mat.shape),
    classifier='RandomForestClassifier',
    n_estimators=int(rf_model.n_estimators),
    max_depth=int(rf_model.max_depth),
    class_weight=None,
    dataset='5v_cleandf_RData',
    features=FEATURES,
    tier_labels=TIER_LABELS,
    total_samples=len(esi_all),
    n_red=int(np.sum(y_3tier_all == 0)),
    n_yellow=int(np.sum(y_3tier_all == 1)),
    n_green=int(np.sum(y_3tier_all == 2)),
    overall_accuracy=round(acc_3tier, 4),
    macro_balanced_accuracy=round(bal_acc_3tier, 4),
    macro_f1=round(macro_f1, 4),
    holdout_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'lmnn_random_forest_3class_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported 3-Class LMNN + Random Forest Bundle  : {bundle_file}")
print(f"✓ Exported 3-Class LMNN + Random Forest Manifest: {manifest_file}")